# Workbench Notebook

This notebook runs in the dedicated `workbench-jupyter` Conda environment.

Use the `wbn` helper below to call Workbench data-router endpoints without putting provider credentials in notebook code.

In [ ]:
import requests

try:
    import pandas as pd
except Exception:
    pd = None

class _Wbn:
    _BASE = "http://127.0.0.1:4000"

    def query(self, moniker, params=None):
        response = requests.post(
            f"{self._BASE}/api/data/query",
            json={"moniker": moniker, "params": params or {}},
            timeout=20,
        )
        response.raise_for_status()
        return self._frame(response.json().get("results", []))

    def _query(self, moniker, params=None):
        return self.query(moniker, params)

    def _frame(self, rows):
        return pd.DataFrame(rows) if pd is not None else rows

    def fred(self, symbol, range="1y"):
        return self.query(
            f"macro.indicators/{symbol}",
            {"symbol": symbol, "range": range},
        )

    def equity(self, symbol, range="1y"):
        return self.query(
            f"equity.prices/{symbol}",
            {"range": range},
        )

    def curve(self):
        return self.query("fixed.income.govies")

    def rates(self):
        return self.query("reference.rates")

    def snapshot(self):
        return self.query("macro.indicators")

wbn = _Wbn()
print("wbn ready")

In [ ]:
wbn.query("fixed.income.govies")

In [ ]:
# Workbench query: fixed.income.govies
df = wbn.query("fixed.income.govies")
df